In [1]:
import time
import gurobipy as gb
import numpy as np
import csv

In [2]:
# Class which can have attributes set.
class expando(object):
    pass

In [3]:
#Adding Feasibility Cuts
class Lifted_NISSP:
    def __init__(self, MP, scenario):
        self.data = expando()
        self.variables = expando()
        self.constraints = expando()
        self.results = expando()
        
        self.MP = MP
        self.data.scenario = scenario
        self._build_model()
      

    def optimize(self):
        self.model.optimize()
        
    def _build_model(self):
        self.model = gb.Model()
        self._build_variables()
        self._build_objective()
        self._build_constraints()
        self.model.update()
        
    def _build_variables(self):
        m = self.model
        self.variables.w_X_osrt = m.addVars([(b, c, d) for b in range(self.MP.data.S) for c in range(self.MP.data.R) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="w_X_osrt")
        self.variables.w_I_ont1t = m.addVars([(b, c, d) for b in range(self.MP.data.N) for c in range(self.MP.data.T) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="w_I_ort1t")
        self.variables.X_snt = m.addVars([(a, b, c) for a in range(self.MP.data.S) for b in range(self.MP.data.N) for c in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="X_snt")
        self.variables.Y_onrt1t = m.addVars([(a, b, c, d) for a in range(self.MP.data.N) for b in range(self.MP.data.R) for c in range(self.MP.data.T) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="Y_nrt1t")
        self.variables.phi_ort = m.addVars([(b, c) for b in range(self.MP.data.R) for c in range(self.MP.data.T)], vtype=gb.GRB.CONTINUOUS, name="phi_ort")
        m.update()
        
    def _build_objective(self):
        m = self.model
        obj = 0
        obj += sum(self.MP.data.w_C1*self.variables.w_I_ont1t[(n,t1,t)] for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) for t in range(self.MP.data.T) if self.MP.data.tau <= t - t1 < self.MP.data.gama)
        obj += sum(self.MP.data.h_C_sr[s][r]*self.variables.w_X_osrt[(s,r,t)] for s in range(self.MP.data.S) for r in range(self.MP.data.R) for t in range(self.MP.data.T)) 
        obj += sum(self.variables.Y_onrt1t[(n,r,t1,t)]*self.MP.data.w_C_nr[n][r] for n in range(self.MP.data.N) for r in range(self.MP.data.R) for t in range(self.MP.data.T) for t1 in range(self.MP.data.T))
        obj += sum(self.MP.data.delta_o*self.variables.phi_ort[(r,t)] for r in range(self.MP.data.R) for t in range(self.MP.data.T))
        self.model.setObjective(obj, gb.GRB.MINIMIZE)
        
    def _build_constraints(self):
        m = self.model
        self.constraints.c1 = m.addConstrs(sum(self.variables.w_X_osrt[(s,r,t)] for r in range(self.MP.data.R)) <= 36500 for s in range(self.MP.data.S) for t in range(self.MP.data.T))
        self.constraints.c2 = m.addConstrs(self.variables.w_I_ont1t[(n,t1,t)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for r in range(self.MP.data.R)) == self.MP.data.rho1*sum(self.variables.X_snt[(s,n,t1)] for s in range(self.MP.data.S)) for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) for t in range(self.MP.data.T) if t-t1 == self.MP.data.tau)
        self.constraints.c3 = m.addConstrs(self.variables.w_I_ont1t[(n,t1,t)] - self.MP.data.rho*self.variables.w_I_ont1t[(n,t1,t-1)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for r in range(self.MP.data.R)) == 0 for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) for t in range(self.MP.data.T) if self.MP.data.tau < t-t1 < self.MP.data.gama)
        self.constraints.c4 = m.addConstrs(self.variables.phi_ort[(r,t)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) if self.MP.data.tau <= t-t1 < self.MP.data.gama) + sum(self.variables.w_X_osrt[(s,r,t)] for s in range(self.MP.data.S)) >= self.MP.data.demand[self.data.scenario][r][t] for r in range(self.MP.data.R) for t in range(self.MP.data.T))
        self.constraints.c5 = m.addConstrs(sum(self.variables.w_I_ont1t[(n,t1,t)] for t1 in range(self.MP.data.T)) <= self.MP.data.w_alpha for n in range(self.MP.data.N) for t in range(self.MP.data.T))
        self.constraints.c6 = m.addConstrs((sum(self.variables.X_snt[(s,n,t)] for n in range(self.MP.data.N)) <= 40000 for s in range(self.MP.data.S) for t in range(self.MP.data.T)))
        

In [4]:
# Master problem
class Relaxed_Master:
    def __init__(self,count,ep=0.000001, de=0.0001,scenarios=100):
        self.data = expando()
        self.variables = expando()
        self.constraints = expando()
        self.params = expando()
        self.params.scenarios = scenarios
        self.params.i = count
        self._load_data()
        self._build_model()
        self._init_benders_params(ep=ep, de=de)
        self._start_from_previous()
    
    def optimize(self):
        self.model.Params.lazyConstraints = 1
        #self.model.Params.Threads = 1
        #start1 = time.time()
        
        h = np.zeros(self.data.O)
        
        for o in range(self.data.O):
               
            sm = Lifted_NISSP(self, o)
            sm.model.setParam ( "OutputFlag", 0 )
            sm.optimize()
            self.model.addConstr(self.variables.theta >= sm.model.ObjVal)
            
        self.model.update()          
        self.x = np.zeros((self.data.S,self.data.N,self.data.T))
        self.theta = 0
        self.sub = np.zeros(self.data.O)
        self.scene = {}
        start2 = time.time()
        while ((abs(self.data.ub-self.data.lb)) > 0.000001*self.data.lb):
            
            self.model.setParam ( "OutputFlag", 0 )
            self.model.optimize()        
            self.z_master = self.model.ObjVal
            
            for s in range(self.data.S):
                for n in range(self.data.N):
                    for t in range(self.data.T):
                        self.x[s][n][t] = self.variables.X_snt[(s,n,t)].x
            
            self.theta = self.variables.theta.x
                
            for scenario in range(self.params.scenarios):    
                sm = Subproblem_NISSPo(self,scenario)
                for n in range(self.data.N):
                    for t1 in range(self.data.T):
                        for t in range(self.data.T):
                            if (t - t1 == self.data.tau):
                                sm.constraints.c2[(n,t1,t)].rhs = sum(self.data.rho1*self.x[s][n][t1] for s in range(m.data.S))
                        
                obj = sum(self.data.w_C1*sm.variables.w_I_ont1t[(n,t1,t)] for n in range(self.data.N) for t1 in range(self.data.T) for t in range(self.data.T))
                obj += sum(sm.variables.Y_onrt1t[(n,r,t1,t)]*self.data.w_C_nr[n][r] for n in range(self.data.N) for r in range(self.data.R) for t in range(self.data.T) for t1 in range(self.data.T))
                obj += sum(self.data.h_C_sr[s][r]*sm.variables.w_X_osrt[(s,r,t)] for s in range(self.data.S) for r in range(self.data.R) for t in range(self.data.T))     
                obj += sum(self.data.delta_o*sm.variables.phi_ort[(r,t)] for r in range(self.data.R) for t in range(self.data.T))
                sm.model.setObjective(obj, gb.GRB.MINIMIZE) 
                sm.model.setParam ( "OutputFlag", 0 )
                
                sm.model.optimize()
                
                self.sub[scenario] = sm.model.ObjVal    
                self.scene[scenario] = sm        
                pi1 = np.zeros((self.data.S, self.data.T))
                pi2 = np.zeros((self.data.N, self.data.T, self.data.T))
                pi3 = np.zeros((self.data.N, self.data.T, self.data.T))
                pi4 = np.zeros((self.data.R, self.data.T))
                pi5 = np.zeros((self.data.N, self.data.T))
        
                for t in range(self.data.T):
                    for s in range(self.data.S):
                        pi1[(s,t)] = sm.constraints.c1[(s,t)].pi
                    for n in range(self.data.N):
                        for t1 in range(self.data.T):
                            if (t - t1 == self.data.tau):
                                pi2[(n,t1,t)] = sm.constraints.c2[(n,t1,t)].pi
                            if (self.data.tau<t-t1<self.data.gama):
                                pi3[(n,t1,t)] = sm.constraints.c3[(n,t1,t)].pi
                        pi5[(n,t)] = sm.constraints.c5[(n,t)].pi 
                    for r in range(self.data.R):
                        pi4[(r,t)] = sm.constraints.c4[(r,t)].pi
                                    
                
                if (self.sub[scenario] > self.theta):
                    self.model.addConstr(self.variables.theta >= 
                                (sum(pi1[(s,t)] for s in range(self.data.S) for t in range(self.data.T))*36500) +
                                (sum(sum(self.variables.X_snt[(s,n,t1)] for s in range(self.data.S))*self.data.rho1*pi2[(n,t1,t)] for n in range(self.data.N) for t1 in range(self.data.T) for t in range(self.data.T))) +
                                (sum(self.data.demand[scenario][r][t]*pi4[(r,t)] for r in range(self.data.R) for t in range(self.data.T))) +
                                (sum(self.data.w_alpha*pi5[(n,t)] for r in range(self.data.N) for t in range(self.data.T))))
                    
            self.model.update()
            
            if (True):
                th = 0
                index = 0
                for o in range(self.data.O):
                    if (th < self.sub[o]):
                        th = self.sub[o]
                        index = o
                FC = 0
                IC1 = 0
                self.data.X = 0
                self.data.Xs = self.x
                for t in range(self.data.T):
                    for n in range(self.data.N):
                        for s in range(self.data.S):
                            FC += self.x[(s, n, t)]*self.data.C_sn[s][n]
                            self.data.X += self.x[(s, n, t)]
                self.data.FC = FC 
                self.data.PC = 0
                self.data.IC = 0
                self.data.EC = 0
                self.data.TC = 0
                for o in range(m.data.O):
                    self.data.Sh[0][o] = 0
                    self.data.In[0][o] = 0
                    self.data.Em[0][o] = 0
                    self.data.Tr[0][o] = 0
                    Em = 0
                    Tr = 0
                    for t in range(self.data.T):
                        for t1 in range(self.data.T):
                            for n in range(self.data.N):
                                self.data.In[0][o] += self.scene[o].variables.w_I_ont1t[(n,t1,t)].x
                                if (o == index):
                                    self.data.IC += self.data.w_C1*self.scene[o].variables.w_I_ont1t[(n,t1,t)].x
                        for r in range(m.data.R):
                            self.data.Sh[0][o] += self.scene[o].variables.phi_ort[(r,t)].x
                            if (o == index):
                                self.data.PC += self.data.delta_o*self.scene[o].variables.phi_ort[(r,t)].x
                            for t1 in range(self.data.T):
                                for n in range(self.data.N):
                                    self.data.Tr[0][o] += self.scene[o].variables.Y_onrt1t[(n,r,t1,t)].x
                                    Tr += self.scene[o].variables.Y_onrt1t[(n,r,t1,t)].x*self.data.w_C_nr[n][r]
                                    if (o == index):
                                        self.data.TC += self.scene[o].variables.Y_onrt1t[(n,r,t1,t)].x*self.data.w_C_nr[n][r]
                            for s in range(self.data.S):
                                self.data.Em[0][o] += self.scene[o].variables.w_X_osrt[(s,r,t)].x
                                Em += self.scene[o].variables.w_X_osrt[(s,r,t)].x*self.data.h_C_sr[s][r]
                                if (o == index):
                                    self.data.EC += self.data.h_C_sr[s][r]*self.scene[o].variables.w_X_osrt[(s,r,t)].x
                    self.data.Tot[0][o] = self.data.FC + self.data.w_C1*self.data.In[0][o]  + self.data.delta_o*self.data.Sh[0][o] + Em + Tr
                self.data.ToC = self.data.FC + self.data.PC + self.data.IC + self.data.EC + self.data.TC
                self._update_bounds()
                print('* Benders\' step {0}:'.format(len(self.data.upper_bounds)), file=open("output15-100-"+str(self.params.i+1)+".txt", "a"))
                print('* Upper bound: {0}'.format(self.data.ub), file=open("output15-100-"+str(self.params.i+1)+".txt", "a"))
                print('* Lower bound: {0}'.format(self.data.lb), file=open("output15-100-"+str(self.params.i+1)+".txt", "a"))
                print('* Relative Gap: {0}'.format(((self.data.ub - self.data.lb)*100)/self.data.lb), file=open("output15-100-"+str(self.params.i+1)+".txt", "a"))
                end2 = time.time()
                time1 = end2-start2
                print('* Time: {0}'.format(time1), file=open("output15-100-"+str(self.params.i+1)+".txt", "a"))

            
    def _init_benders_params(self, ep=0.000001, de=0.000001):
        self.data.upper_bounds = []
        self.data.lower_bounds = []
        self.data.ub = gb.GRB.INFINITY
        self.data.lb = -gb.GRB.INFINITY 
        self.data.Theta = 0
        self.data.V = 0
        self.data.Zi = 0
        self.data.Z_eta = 0
        self.data.phi_ort = np.zeros((self.data.O, self.data.R, self.data.T))
    
    def _load_data(self):
        self.data.S = 4
        self.data.N = 6
        self.data.R = 10
        self.data.T = 9
        self.data.O = self.params.scenarios
        self.data.FC = 0
        self.data.X = 0
        self.data.TC = 0
        self.data.EC = 0
        self.data.IC = 0
        self.data.PC = 0
        self.data.ToC = 0
        self.data.Xs = np.zeros((self.data.S,self.data.N,self.data.T))
        self.data.Sh = np.zeros((1,self.data.O))
        self.data.Tot = np.zeros((1,self.data.O))
        self.data.In = np.zeros((1,self.data.O))
        self.data.Em = np.zeros((1,self.data.O))
        self.data.Tr = np.zeros((1,self.data.O))
        self.data.eta = 0
        self.data.extra = 0
        fil_name = 'beta_st20'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.beta_st = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.beta_st.append(nwrow)
            
        fil_name = 'w_beta_st20'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.w_beta_st = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.w_beta_st.append(nwrow)
            
        fil_name = 'C_sn20'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.C_sn = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.C_sn.append(nwrow)
            
        fil_name = 'w_C_nr20'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.w_C_nr = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.w_C_nr.append(nwrow)
            
        fil_name = 'h_C_sr20'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.h_C_sr = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.h_C_sr.append(nwrow)    
        """   
        fil_name = 'Xs'
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.X = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.X.append(nwrow)                
        """   
        fil_name = 'demand-'+str(self.params.i+1)
        with open(fil_name+'.csv', 'r') as f:
            reader = csv.reader(f)
            examples = list(reader)
        self.data.demand = []
        for row in examples:
            nwrow = []
            for r in row:
                nwrow.append(eval(r))
            self.data.demand.append(nwrow)
         
        
        self.data.rho = 1
        self.data.rho1 = 0.8
        self.data.w_C1 = 200
        self.data.tau = 1
        self.data.gama = 4
        self.data.w_alpha = 50000
        self.data.p_o = 1/self.data.O
        self.data.delta_o = 9500
        
        
    def _build_model(self):
        self.model = gb.Model()
        self._build_variables()
        self._build_objective()
        self._build_constraints()
        self.model.update()  
        
    def _build_variables(self):
        m = self.model
        self.variables.X_snt = m.addVars([(a, b, c) for a in range(self.data.S) for b in range(self.data.N) for c in range(self.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="X_snt")
        self.variables.theta = m.addVar(lb = 0, vtype=gb.GRB.CONTINUOUS, name="theta") 
        m.update()
        
    def _build_objective(self):
        obj = 0
        obj += sum(sum(self.variables.X_snt[(s, n, t)] for t in range(self.data.T))*self.data.C_sn[s][n] for s in range(self.data.S) for n in range(self.data.N))
        obj += self.variables.theta
        
        self.model.setObjective(obj, gb.GRB.MINIMIZE)
        
    def _build_constraints(self):
        m = self.model
        self.constraints.c1 = m.addConstrs((sum(self.variables.X_snt[(s,n,t)] for n in range(self.data.N)) <= 40000 for s in range(self.data.S) for t in range(self.data.T)))
        #self.constraints.c2 = m.addConstrs((self.variables.X_snt[(s,n,t)] == self.data.X[s][n][t] for s in range(self.data.S) for n in range(self.data.N) for t in range(self.data.T)))
        pass
    
    def _update_bounds(self):
        z_sub = 0
        theta = 0
        self.data.lb = self.z_master
        for o in range(self.data.O):
            if (z_sub < self.sub[o]):
                z_sub = self.sub[o]
        theta = self.theta
        
        temp = self.z_master - theta + z_sub 

        if (temp < self.data.ub):
            self.data.ub = temp
        
        self.data.upper_bounds.append(self.data.ub)
        self.data.lower_bounds.append(self.data.lb)
    
    def _start_from_previous(self):
        '''
            Used to warm-start MIP problems.
        '''
        pass


In [5]:
# Subproblem-2
class Subproblem_NISSPo:
    def __init__(self, MP, scenario):
        self.data = expando()
        self.variables = expando()
        self.constraints = expando()
        self.results = expando()
        self.MP = MP
        self.data.scenario = scenario
        self._build_model()

    def optimize(self):
        self.model.optimize()
        
    def _build_model(self):
        self.model = gb.Model()
        self._build_variables()
        self._build_objective()
        self._build_constraints()
        self.model.update()
        
    def _build_variables(self):
        m = self.model
        self.variables.w_X_osrt = m.addVars([(b, c, d) for b in range(self.MP.data.S) for c in range(self.MP.data.R) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="w_X_osrt")
        self.variables.w_I_ont1t = m.addVars([(b, c, d) for b in range(self.MP.data.N) for c in range(self.MP.data.T) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="w_I_ort1t")
        self.variables.Y_onrt1t = m.addVars([(a,b, c, d) for a in range(self.MP.data.N) for b in range(self.MP.data.R) for c in range(self.MP.data.T) for d in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="Z_ort1t")
        self.variables.phi_ort = m.addVars([(b, c) for b in range(self.MP.data.R) for c in range(self.MP.data.T)], lb = 0, vtype=gb.GRB.CONTINUOUS, name="phi_ort")
        m.update()
        
    def _build_objective(self):
        m = self.model
        obj = 0
        self.model.setObjective(obj, gb.GRB.MINIMIZE)
    
    def _build_constraints(self):
        m = self.model
        self.constraints.c1 = m.addConstrs(sum(self.variables.w_X_osrt[(s,r,t)] for r in range(self.MP.data.R)) <= 36500 for s in range(self.MP.data.S) for t in range(self.MP.data.T))
        self.constraints.c2 = m.addConstrs(self.variables.w_I_ont1t[(n,t1,t)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for r in range(self.MP.data.R)) == 0 for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) for t in range(self.MP.data.T) if t-t1 == self.MP.data.tau)
        self.constraints.c3 = m.addConstrs(self.variables.w_I_ont1t[(n,t1,t)] - self.MP.data.rho*self.variables.w_I_ont1t[(n,t1,t-1)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for r in range(self.MP.data.R)) == 0 for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) for t in range(self.MP.data.T) if self.MP.data.tau < t-t1 < self.MP.data.gama)
        self.constraints.c4 = m.addConstrs(self.variables.phi_ort[(r,t)] + sum(self.variables.Y_onrt1t[(n,r,t1,t)] for n in range(self.MP.data.N) for t1 in range(self.MP.data.T) if self.MP.data.tau <= t-t1 < self.MP.data.gama) + sum(self.variables.w_X_osrt[(s,r,t)] for s in range(self.MP.data.S)) >= self.MP.data.demand[self.data.scenario][r][t] for r in range(self.MP.data.R) for t in range(self.MP.data.T))
        self.constraints.c5 = m.addConstrs(sum(self.variables.w_I_ont1t[(n,t1,t)] for t1 in range(self.MP.data.T)) <= self.MP.data.w_alpha for n in range(self.MP.data.N) for t in range(self.MP.data.T))
        

In [ ]:
for i in range(1):
    m = Relaxed_Master(i)
    m.optimize()
    print ("Total:" + str(m.data.ToC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("FC:" + str(m.data.FC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("TC:" + str(m.data.TC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("IC:" + str(m.data.IC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("EC:" + str(m.data.EC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("PC:" + str(m.data.PC), file=open("output16-800-"+str(i+1)+".txt", "a"))
    print ("FS:" + str(m.data.X), file=open("output16-800-"+str(i+1)+".txt", "a"))
"""    
    fil_name = 'Tot'
    example = m.data.Tot
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)  
    fil_name = 'Inv'
    example = m.data.In
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)  
    fil_name = 'Emg'
    example = m.data.Em
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)  
    fil_name = 'Sho'
    example = m.data.Sh
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)          
    fil_name = 'Tr'
    example = m.data.Tr
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)     
    fil_name = 'Xs'
    example = m.data.Xs
    example = example.tolist()
    with open(fil_name+'.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=',')
        writer.writerows(example)             
"""    

Using license file /software/commercial/gurobi/gurobi.lic
Set parameter TokenServer to value license8.clemson.edu
Changed value of parameter lazyConstraints to 1
   Prev: 0  Min: 0  Max: 1  Default: 0
